# Certify the fault distance of circuits

Brute-force fault-distance enumeration is expensive because the number of fault combinations grows rapidly.
`circuits.get_fault_distance` converts a noisy Stim circuit into a detector error model, maps it to an auxiliary CSS code, and uses the Brouwer-Zimmermann code-distance algorithm to find the fewest faults that flip an observable without firing a detector.

For the full construction and mathematical details, see `help(circuits.get_fault_distance)`.

The examples below certify the distance of a surface-code memory experiment and a flagged Steane-state preparation circuit.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import itertools

import stim

from qldpc import circuits, codes

## Certify fault distance in a memory experiment

We use a distance-5 surface code with five rounds of syndrome extraction, testing X- and Z-basis memory separately.

We use Stim's graphlike search as a cross-check for this surface-code example because each fault in its syndrome-extraction circuit flips at most two detectors.
General memory experiments can contain non-graphlike faults, for which Stim's graphlike search need not identify the true fault distance.
Using `circuits.get_fault_distance` does not require graphlike faults.

In [3]:
code = codes.SurfaceCode(5)
noise_model = circuits.DepolarizingNoiseModel(1e-3)

for basis in ("X", "Z"):
    # one logical memory cycle = code.get_distance() rounds of syndrome extraction
    # you can replace logical_memory_cycle with another memory circuit
    initialization, logical_memory_cycle, readout, *_ = circuits.get_memory_experiment_parts(
        code, basis=basis, num_rounds=code.get_distance()
    )

    # keep initialization and readout noiseless, since we are certifying the fault-distance
    # of the inner QEC cycle
    noisy_circuit = initialization + noise_model.noisy_circuit(logical_memory_cycle) + readout
    fault_distance = circuits.get_fault_distance(noisy_circuit)
    graphlike_distance = len(noisy_circuit.shortest_graphlike_error())
    assert fault_distance == graphlike_distance
    print(
        f"{basis}-basis memory: fault distance {fault_distance}, "
        f"Stim graphlike distance {graphlike_distance}"
    )

X-basis memory: fault distance 5, Stim graphlike distance 5
Z-basis memory: fault distance 5, Stim graphlike distance 5


## Certify fault-tolerant state preparation

Fault distance identifies the smallest undetected logical fault pattern.
For a prepared resource state, we also need to control residual errors that can propagate into other code blocks.
We require every accepted pattern of $r$ preparation faults, up to a chosen budget, to leave an error equivalent on the intended state to one of weight at most $r$.
The choice of weight criterion must reflect the resource's downstream use.

### Undetected logical fault distance

We now compute the fault distance of a circuit that prepares the Steane code in $|0_L\rangle$, using the circuit from the [state-preparation notebook](5_state_preparation.ipynb).
Every measurement in the preparation circuit is post-selected: a nonzero outcome rejects the attempt and restarts preparation.
Here the only such measurement is the final X-basis measurement of qubit 7.
The flag checks a logical-Z representative, so it can reveal logical-X faults in the preparation.

In [4]:
code = codes.SteaneCode()

state_prep_circuit = stim.Circuit("""
    # non-fault-tolerant state preparation
    H 0 2 4
    CX 0 3 2 1 4 5
    CX 0 1 2 6 4 3
    CX 2 5 3 6
    # flag errors in a logical Z representative
    H 7
    CZ 7 1 7 3 7 5
    MX 7
""")

`get_state_prep_diagnostic_circuit` adds a flag detector for the preparation measurement, noiseless final stabilizer measurements, and a noiseless logical-Z measurement.
The logical-Z observable is inferred automatically from the prepared state.
The diagnostic operations remain noiseless when we apply the noise model.

For the undetected logical fault distance, we require both the flag and the final stabilizer syndrome to be zero while the logical-Z observable flips.

In [5]:
diagnostic_circuit, _ = circuits.get_state_prep_diagnostic_circuit(
    code,
    state_prep_circuit,
    add_flags=True,
)
noisy_circuit = noise_model.noisy_circuit(diagnostic_circuit)
fault_distance = circuits.get_fault_distance(noisy_circuit)
print(f"State-preparation undetected logical fault distance: {fault_distance}")

State-preparation undetected logical fault distance: 3


### Bounded error spread

We begin with Z errors in a CSS logical-zero state, then consider joint X/Z errors and post-selection.

Consider a binary CSS stabilizer code with all $k$ logical qubits prepared in $|0\rangle$.
Write its X- and Z-check matrices as $H_X$ and $H_Z$, and its logical-Z generators as $L_Z$.
Both the Z checks and the logical-Z operators stabilize the prepared state.
The vectors representing harmless Z operators therefore form

$$
S_Z = \operatorname{rowspan}\begin{pmatrix} H_Z \\ L_Z \end{pmatrix} = \ker H_X.
$$

Let $z$ represent a Z error on the physical data qubits.
Its effective weight is the minimum weight of a representative with the same effect on the prepared state:

$$
w_{\mathrm{eff},Z}(z)
= \min_{s\in S_Z} |z+s|
= \min_{H_X \cdot z'=H_X \cdot z} |z'|.
$$

Thus the effective Z weight depends only on the X-check syndrome $H_X \cdot z$.
Equivalent errors produce the same prepared state, so subsequent processing preserves their equivalence.
A weight-$w$ representative propagates to at most $w$ qubits per block under a transversal coupling.

#### Compare worst-case effective weights

The maximum effective Z weight is the covering radius of the classical code $\ker H_X$:

$$
R_Z = \max_z w_{\mathrm{eff},Z}(z).
$$

We compute this code-dependent quantity for the Steane code and a distance-3 surface code by enumerating their output Z patterns.

In [6]:
for example_code in (codes.SteaneCode(), codes.SurfaceCode(3)):
    example_harmless_z = codes.ClassicalCode(example_code.matrix_x)
    max_effective_z_weight = max(
        example_harmless_z.get_distance(vector=example_code.field(bits))
        for bits in itertools.product((0, 1), repeat=len(example_code))
    )
    print(f"{example_code.name}: maximum effective Z weight {max_effective_z_weight}")

SteaneCode: maximum effective Z weight 1
SurfaceCode: maximum effective Z weight 2


#### Choose a joint error-weight criterion

A pure-Z calculation is not a certificate for mixed Pauli faults.
Let $T$ be the commutation map of the complete independent stabilizers of the intended pure output state, including its logical stabilizers.
For a physical Pauli vector $e=(x\mid z)$, distinguish support weight $w_{\rm Pauli}(e)=|\operatorname{supp}(x)\cup\operatorname{supp}(z)|$ from CSS-sector weight $w_{\rm CSS}(e)=\max(|x|,|z|)$.
Define the effective chosen weight by

$$
\mu_w(s)=\min_{Te=s} w(e).
$$

Both metrics count Y once, but $X_iZ_j$ on distinct qubits has physical weight two and CSS weight one.
The minimization is joint over one representative; independently minimizing its X and Z projections can be wrong for a non-CSS target state.
With `css_weight=None`, the complete prepared-state stabilizer row space selects CSS weight exactly when its pure-X and pure-Z intersections span the whole space in the physical output basis.
The test is independent of the chosen generators and does not use local-Clifford equivalence.
It establishes state equivalences, not downstream compatibility: a CSS code can prepare a non-CSS logical state, such as a logical Y eigenstate, whose sector-wise application needs an explicit override.
Use `css_weight=True` for CSS-sector weight or `css_weight=False` for physical support; the result records the resolved `css_weight` boolean.
No code argument is required or accepted.
Equal complete syndromes mean errors differ by a state stabilizer, because a pure-state stabilizer space is maximal isotropic.
`circuits.certify_state_prep_spread` compiles actual rejection flags $P$ and separate noiseless final diagnostics $S$, and exactly checks the following for a budget of $t$ faults (`max_faults`):

$$
Pf=0,\quad 1\le |f|\le t\quad\Longrightarrow\quad\mu_w(Sf)\le |f|.
$$

Here $|f|$ counts unit-cost binary DEM mechanisms, not mutually exclusive physical gate alternatives.
Correlated mechanisms stay intact, and flagged mechanisms are not deleted: two flags can cancel.
The effective weight is subadditive, so it suffices to check support-minimal accepted patterns.
Such a pattern on $J$ has $\operatorname{nullity}(P[:,J])=1$ and hence $|J|\le\operatorname{rank}(P)+1$.
The search combines this reduction with adaptive independent-prefix closure or packed Brouwer–Zimmermann coverage and amortized affine Pauli-coset threshold proofs, rather than enumerating every fault combination.
Canonical closure matches an independent prefix to its XOR closing column, generating each minimal dependence once.
With independent distinct nonzero flag types, acceptance forces even multiplicity of each type, so only equal-flag pairs are minimal even at larger budgets.
The returned certificate includes the diagnostic DEM and either a completed decision or a concrete accepted counterexample with a proved effective-weight lower bound.
Random post-selected branches, mixed outputs, and nondeterministic existing annotations are explicitly unsupported.

In [7]:
flagged_preparation = state_prep_circuit.copy()
flagged_preparation.append("DETECTOR", [stim.target_rec(-1)])
noisy_preparation = noise_model.noisy_circuit(flagged_preparation)
spread = circuits.certify_state_prep_spread(
    noisy_preparation,
    output_qubits=range(7),
    max_faults=3,
    post_select=[0],
)
assert spread.certified and spread.css_weight is True
physical = circuits.certify_state_prep_spread(
    noisy_preparation, output_qubits=range(7), max_faults=3, post_select=[0], css_weight=False
)
witness = physical.counterexample
assert not physical.certified and witness is not None
assert witness.fault_count == 1 and witness.effective_weight_lower_bound == 2
css_representative = stim.PauliString("X_Z____")
assert (
    tuple(
        int(not css_representative.commutes(stim.PauliString(g)))
        for g in physical.target_stabilizers
    )
    == witness.output_syndrome
)
print(f"CSS-sector criterion (css_weight={spread.css_weight}): {spread.certified}")
print(f"Physical-support criterion (css_weight={physical.css_weight}): {physical.certified}")
print("Y0 Z1 is equivalent to X0 Z2: CSS weight 1, physical support 2")

CSS-sector criterion (css_weight=True): True
Physical-support criterion (css_weight=False): False
Y0 Z1 is equivalent to X0 Z2: CSS weight 1, physical support 2


For the complete stabilizers returned here, in order,

```
+X_X_X_X  +_XX__XX  +___XXXX  +Z____ZZ  +_Z__Z_Z  +__Z_ZZ_  +___ZZZZ
```

the accepted DEM mechanism `D1 D2 D4` has syndrome $(1,1,0,1,0,0,0)$ and no actual flag event `D0`.
One physical realization is $Y_0Z_1$ immediately after `CX 0 1` in the second encoder layer.
It commutes with all remaining encoder gates and with the subsequent verifier's data-Z couplings, so the output is still $Y_0Z_1$ and the flag is unchanged.
Its last four syndrome bits equal the X syndrome of qubit 0, which is distinct from every other qubit's X syndrome.
Thus a weight-one representative with those last four bits would have to be $X_0$ or $Y_0$.
Their first three bits are $(0,0,0)$ and $(1,0,0)$, not $(1,1,0)$.
A Z error has zero last-four syndrome bits.
Therefore no weight-zero or weight-one Pauli has this syndrome, while $Y_0Z_1$ has weight two: the effective weight is exactly two.
This algebraic witness refutes the physical-support criterion only.
It does not refute the CSS-sector criterion: $Y_0Z_1$ and $X_0Z_2$ differ by the prepared-state stabilizer $Z_0Z_1Z_2$, up to global phase.
The latter has one X and one Z, so its CSS weight is one.
The automatically selected CSS-sector calculation checks every required accepted pattern and certifies the condition through the requested budget.
The physical-support failure reflects a stricter metric, not proof that this preparation fails the CSS application.

The following controlled model illustrates genuine three-mechanism dependencies with multiple independent rejection flags.

In [8]:
encoder = stim.Circuit("""
    H 0 2 4
    CX 0 3 2 1 4 5
    CX 0 1 2 6 4 3
    CX 2 5 3 6
""")
controlled = stim.Circuit()
for start in (0, 7, 14):
    controlled += circuits.with_remapped_qubits(encoder, range(start, start + 7))
data_error = [
    target for start in (0, 7, 14) for target in (stim.target_y(start), stim.target_z(start + 1))
]
for group in range(4):
    a, b = 21 + 2 * group, 22 + 2 * group
    controlled.append("CORRELATED_ERROR", [stim.target_x(a)], 1e-3)
    controlled.append("CORRELATED_ERROR", [stim.target_x(b)], 1e-3)
    controlled.append("CORRELATED_ERROR", data_error + [stim.target_x(a), stim.target_x(b)], 1e-3)
controlled.append("M", range(21, 29))
for j in range(8):
    controlled.append("DETECTOR", [stim.target_rec(j - 8)])
many_flags = circuits.certify_state_prep_spread(
    controlled, output_qubits=range(21), max_faults=3, post_select=range(8)
)
assert many_flags.certified and many_flags.css_weight is True
assert many_flags.statistics.flag_rank == 8
print(
    f"css_weight={many_flags.css_weight}: flag rank 8, budget 3, certified={many_flags.certified}"
)

css_weight=True: flag rank 8, budget 3, certified=True


Each flag pair has signatures $a,b,a+b$, so the full triple is accepted while no nonzero proper subset is accepted.
There are four such dependencies and eight independent rejection flags.
The accepted triple applies $Y_0Z_1$ in each Steane block, equivalent to $X_0Z_2$ in each block.
That representative has three X and three Z components, giving CSS weight three, equal to its three-fault cost.
The model's other accepted patterns decompose into these minimal triples, so subadditivity proves the bounded condition.
This is a controlled binary DEM example, not a claim that the noiseless encoder or every physical noise model is fault-tolerant.

The code-agnostic physical-support criterion remains available with `css_weight=False`, including for non-CSS resource states.

In [9]:
resource = stim.Circuit("H 0 1 2 3 4\nCZ 0 1 1 2 2 3 3 4 4 0\nS 0 2 4")
resource.append("DEPOLARIZE1", range(5), 1e-3)
certificate = circuits.certify_state_prep_spread(
    resource, output_qubits=range(5), max_faults=3, css_weight=False
)
assert certificate.certified and certificate.css_weight is False
print(f"Physical support certified through {certificate.max_faults} faults: True")

Physical support certified through 3 faults: True


A completed bounded-spread certificate is a sufficient resource-state guarantee, not an arbitrary algorithm's fault-tolerance proof.
It supports distance preservation when subsequent fault-tolerant gadgets correct or propagate an input error of weight $r$ and additional faults within the corresponding combined budget.
CSS-sector use requires suitable sector-preserving downstream operations; arbitrary Clifford operations can mix sectors and need not preserve this norm.
Automatic state detection does not calculate a code's correction radius or verify a decoder.
Neither acceptance probabilities nor physical-gate compatibility constraints are established by this binary DEM certificate.
The search is exact but remains exponential in general, with no universal speedup promise.